# ⚡ Smart Energy Grid Optimizer — RDMU Topic 5
Colab reproduction. Concepts: **MDP · Sequential Decision Making · Methods for Estimation · Multicriteria Decision Making** (+ Utility Theory).

Two engines: constraint-programming LP (optimal) and SAC RL agent with constraint-projection.

In [ ]:
!pip -q install cvxpy stable-baselines3 gymnasium plotly pandas

## 1. Clone repo

In [ ]:
# !git clone https://github.com/<your-username>/smartgrid.git
# %cd smartgrid
import sys; sys.path.insert(0,'.')

## 2. Constraint solver (Engine B)

In [ ]:
import pandas as pd
from env.grid_env import GridEnv, SOURCES
from agents.solver import run_episode_solver
cfg={'renewable_penetration':1.0,'demand_variability':0.10,'emission_penalty':40,'emission_cap':400,'seed':7}
df=pd.DataFrame(run_episode_solver(GridEnv(cfg)))
print('Cost  $%.0f'%df['cost'].sum()); print('Emit  %.0f tCO2'%df['emit'].sum())
print('Renew %.0f%%'%(100*df[['d_solar','d_wind']].sum().sum()/df['supply'].sum())); print('Unmet %.1f'%df['unmet'].sum())

## 3. Dispatch stack plot

In [ ]:
import plotly.graph_objects as go
PAL={'solar':'#F6C445','wind':'#4FC3D9','gas':'#E8743B','coal':'#7A6F63','battery':'#9B8CFF'}
fig=go.Figure()
for s in SOURCES: fig.add_trace(go.Scatter(x=df['t'],y=df[f'd_{s}'].clip(lower=0),name=s,stackgroup='one',mode='none',fillcolor=PAL[s]))
fig.add_trace(go.Scatter(x=df['t'],y=df['demand'],name='Demand',line=dict(color='black',dash='dot')))
fig.update_layout(template='plotly_white',height=400,xaxis_title='Hour',yaxis_title='MW'); fig.show()

## 4. Train SAC + projection (Engine A)

In [ ]:
from agents.train_sac import train
model=train(timesteps=8000)  # ~2 min CPU

## 5. RL vs solver

In [ ]:
from agents.rl_agent import run_episode_agent
ad=pd.DataFrame(run_episode_agent(model,cfg,project=True))
def summ(d): return dict(cost=round(d['cost'].sum()),emit=round(d['emit'].sum()),unmet=round(d['unmet'].sum(),1))
print('SOLVER',summ(df)); print('SAC   ',summ(ad))

## 6. Sensitivity — cost vs emissions (Multicriteria / Pareto)

In [ ]:
rows=[]
for p in [0,10,20,40,80,120,200]:
    d=pd.DataFrame(run_episode_solver(GridEnv({**cfg,'emission_penalty':p})))
    rows.append({'penalty':p,'cost':d['cost'].sum(),'emit':d['emit'].sum()})
pf=pd.DataFrame(rows)
fig=go.Figure(go.Scatter(x=pf['emit'],y=pf['cost'],mode='lines+markers+text',text=pf['penalty']))
fig.update_layout(template='plotly_white',xaxis_title='Emissions tCO2',yaxis_title='Cost $',height=400); fig.show()
pf